In [2]:
import pandas as pd
import numpy as np
from pyliftover import LiftOver
from tqdm import tqdm

# ---------- Inputs / Outputs ----------
data_path = '/oak/stanford/groups/mrivas/projects/wgs-constraint-llm/data/'
in_path  = data_path + "scz.tsv.gz"             # your input GWAS-like file
out_path = data_path + "scz_hg38_lifted.tsv.gz" # main output
unmapped_path = data_path + "scz_hg38_unmapped.tsv.gz"  # QC of failures

# ---------- Load & parse ----------
df = pd.read_csv(in_path, sep="\t", dtype={"locus": "string"})
# Expect locus like "1:12345" or "chr1:12345"
parts = df["locus"].str.split(":", n=1, expand=True)
df["chr_hg19"] = parts[0].astype("string")
df["pos_hg19_raw"] = pd.to_numeric(parts[1], errors="coerce")

# Normalize chromosome names to 'chrN'
needs_chr = ~df["chr_hg19"].str.startswith("chr", na=False)
df.loc[needs_chr, "chr_hg19"] = "chr" + df.loc[needs_chr, "chr_hg19"].astype("string")

# ---------- Infer 0- vs 1-based ----------
# pyliftover expects 0-based single-base coordinate inputs, returns 0-based.
lo = LiftOver("hg19", "hg38")

sample = df.loc[df["pos_hg19_raw"].notna(), ["chr_hg19","pos_hg19_raw"]].head(1000).copy()
def count_mapped(offset):
    c = 0
    for chrom, pos in zip(sample["chr_hg19"], sample["pos_hg19_raw"].astype(np.int64) - offset):
        if pos < 0: 
            continue
        m = lo.convert_coordinate(chrom, int(pos))
        if m: c += 1
    return c

mapped_as_0 = count_mapped(offset=0)
mapped_as_1 = count_mapped(offset=1)
use_offset = 0 if mapped_as_0 >= mapped_as_1 else 1
print(f"Detected {'0-based' if use_offset==0 else '1-based'} input (offset={use_offset}).")

# Final hg19 0-based coordinate to feed into liftover
df["pos_hg19"] = (df["pos_hg19_raw"].astype("Int64") - use_offset).astype("Int64")

# ---------- Liftover ----------
# We’ll map one row at a time (fast enough for typical GWAS tables); record metadata
def lift_one(chrom: str, pos0: pd.Interval) -> tuple:
    if chrom is pd.NA or pd.isna(pos0) or pos0 < 0:
        return (pd.NA, pd.NA, pd.NA, 0, "invalid_input")
    hits = lo.convert_coordinate(str(chrom), int(pos0))
    if not hits:
        return (pd.NA, pd.NA, pd.NA, 0, "unmapped")
    # Choose the first mapping (UCSC chains are ordered by best path);
    # each hit is (t_chrom, t_pos0, strand, size)
    t_chrom, t_pos0, strand, _ = hits[0]
    return (t_chrom, int(t_pos0), strand, len(hits), "ok")

tqdm.pandas(desc="Lifting variants")
lifted = df.progress_apply(
    lambda r: lift_one(r["chr_hg19"], r["pos_hg19"]),
    axis=1
)

df["chr_hg38"]   = lifted.map(lambda x: x[0])
df["pos_hg38_0b"] = lifted.map(lambda x: x[1])          # still 0-based
df["strand"]     = lifted.map(lambda x: x[2])
df["n_mappings"] = lifted.map(lambda x: x[3])
df["lift_status"]= lifted.map(lambda x: x[4])

# If you prefer 1-based output for single-base variants, add +1 view:
df["pos_hg38_1b"] = df["pos_hg38_0b"].astype("Int64") + 1

# ---------- Save results ----------
# Keep original columns + new liftover columns, do NOT overwrite source fields.
cols_out = (
    list(df.columns)  # original cols first
    + []              # you can reorder if you want
)
df.to_csv(out_path, sep="\t", index=False, compression="gzip")
print(f"✅ Wrote lifted variants → {out_path}")

# Also write unmapped for inspection
df.loc[df["lift_status"] != "ok"].to_csv(unmapped_path, sep="\t", index=False, compression="gzip")
print(f"ℹ️  Unmapped/invalid rows → {unmapped_path}")

/tmp/ipykernel_174014/2529311595.py:21: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  df.loc[needs_chr, "chr_hg19"] = "chr" + df.loc[needs_chr, "chr_hg19"].astype("string")


Detected 0-based input (offset=0).


Lifting variants: 100%|██████████████████████████████████████████████████| 23087953/23087953 [04:47<00:00, 80292.21it/s]


✅ Wrote lifted variants → /oak/stanford/groups/mrivas/projects/wgs-constraint-llm/data/scz_hg38_lifted.tsv.gz
ℹ️  Unmapped/invalid rows → /oak/stanford/groups/mrivas/projects/wgs-constraint-llm/data/scz_hg38_unmapped.tsv.gz


In [3]:
chk = pd.read_csv(out_path, sep="\t", nrows=5)
print(chk.filter(regex="hg19|hg38|status|n_mappings"))
print("Unmapped rows:", (pd.read_csv(unmapped_path, sep="\t").shape[0]))

  chr_hg19  pos_hg19_raw  pos_hg19 chr_hg38  pos_hg38_0b  n_mappings  \
0     chr1        138484    138484     chr1       138484           1   
1     chr1        138484    138484     chr1       138484           1   
2     chr1        138484    138484     chr1       138484           1   
3     chr1        138484    138484     chr1       138484           1   
4     chr1        138484    138484     chr1       138484           1   

  lift_status  pos_hg38_1b  
0          ok       138485  
1          ok       138485  
2          ok       138485  
3          ok       138485  
4          ok       138485  
Unmapped rows: 2465


In [4]:
import pandas as pd
import numpy as np
from pyliftover import LiftOver
from tqdm import tqdm
import json, ast, os

# ----------- config -----------
data_path = '/oak/stanford/groups/mrivas/projects/wgs-constraint-llm/data/'
in_path   = data_path + "SCHEMA_variant_results.tsv.bgz"         # bgzip-compressed TSV
out_path  = data_path + "SCHEMA_variant_results_hg38.tsv.gz"     # gzip-compressed TSV (bgzip later if needed)
chunksize = 200_000                                  # adjust for your RAM/IO
assume_input_is_0_based = True                       # per your instruction

# ----------- helpers ----------
def parse_alleles(s):
    """Parse alleles column like ["A","G"] or ['A','G'] into (ref, alt)."""
    if pd.isna(s):
        return (pd.NA, pd.NA)
    try:
        a = json.loads(s)
    except Exception:
        try:
            a = ast.literal_eval(s)
        except Exception:
            return (pd.NA, pd.NA)
    if isinstance(a, (list, tuple)) and len(a) >= 2:
        return (str(a[0]), str(a[1]))
    return (pd.NA, pd.NA)

lo = LiftOver("hg19", "hg38")

def liftover_one(chrom, pos0):
    """Return (chr_hg38, pos_hg38_0b, n_mappings, status)."""
    if pd.isna(chrom) or pd.isna(pos0):
        return (pd.NA, pd.NA, 0, "invalid_input")
    try:
        pos0 = int(pos0)
    except Exception:
        return (pd.NA, pd.NA, 0, "invalid_input")
    if pos0 < 0:
        return (pd.NA, pd.NA, 0, "invalid_input")
    hits = lo.convert_coordinate(str(chrom), pos0)
    if not hits:
        return (pd.NA, pd.NA, 0, "unmapped")
    # choose first mapping (UCSC orders by best path)
    tchr, tpos0, strand, _ = hits[0]
    return (tchr, int(tpos0), len(hits), "ok")

# ----------- output prep -----------
if os.path.exists(out_path):
    os.remove(out_path)
wrote_header = False

total = mapped = unmapped = multi = 0

# ----------- stream & lift -----------
for chunk in tqdm(pd.read_csv(in_path, sep="\t", compression="gzip", chunksize=chunksize, dtype="string"),
                  desc="Lifting SCHEMA → hg38"):
    # Parse chr/pos from 'locus' like "1:138484" or "chr1:138484"
    # Keep original columns for output
    if "locus" not in chunk.columns or "alleles" not in chunk.columns:
        raise ValueError("Expected columns 'locus' and 'alleles' not found.")

    # Split locus
    lp = chunk["locus"].str.split(":", n=1, expand=True)
    chunk["chr_hg19"] = lp[0]
    chunk["pos_hg19_raw"] = pd.to_numeric(lp[1], errors="coerce")

    # Normalize chr prefix to 'chrN'
    needs_chr = ~chunk["chr_hg19"].str.startswith("chr", na=False)
    chunk.loc[needs_chr, "chr_hg19"] = "chr" + chunk.loc[needs_chr, "chr_hg19"].astype("string")

    # 0- vs 1-based: per your instruction, assume 0-based input
    offset = 0 if assume_input_is_0_based else 1
    chunk["pos_hg19"] = (chunk["pos_hg19_raw"].astype("Int64") - offset).astype("Int64")

    # Parse ref/alt
    ref_alt = chunk["alleles"].apply(parse_alleles)
    chunk["ref"] = ref_alt.map(lambda x: x[0])
    chunk["alt"] = ref_alt.map(lambda x: x[1])

    # Liftover
    lifted = chunk.apply(lambda r: liftover_one(r["chr_hg19"], r["pos_hg19"]), axis=1)
    chunk["chr_hg38"]    = lifted.map(lambda x: x[0])
    chunk["pos_hg38_0b"] = lifted.map(lambda x: x[1]).astype("Int64")
    chunk["n_mappings"]  = lifted.map(lambda x: x[2]).astype("Int64")
    chunk["lift_status"] = lifted.map(lambda x: x[3])

    # Stats
    total += len(chunk)
    mapped += (chunk["lift_status"] == "ok").sum()
    unmapped += (chunk["lift_status"] == "unmapped").sum()
    multi += (chunk["n_mappings"] > 1).sum()

    # Reorder columns: start with chr,pos,ref,alt (hg38)
    # Keep the rest (original columns) after, with useful audit fields near the front.
    leading = ["chr_hg38", "pos_hg38_0b", "ref", "alt"]
    audit   = ["lift_status", "n_mappings", "chr_hg19", "pos_hg19_raw", "pos_hg19"]
    # Avoid duplicates and maintain original order otherwise
    rest = [c for c in chunk.columns if c not in set(leading + audit)]
    out = chunk[leading + audit + rest].rename(columns={
        "chr_hg38": "chr",
        "pos_hg38_0b": "pos"
    })

    # Write/append
    out.to_csv(out_path, sep="\t", index=False, mode="a",
               header=not wrote_header, compression="gzip")
    wrote_header = True

# ----------- summary -----------
print(f"Done → {out_path}")
print(f"Total rows: {total:,}")
print(f"Mapped:     {mapped:,}  ({mapped/total*100:.3f}%)")
print(f"Unmapped:   {unmapped:,}  ({unmapped/total*100:.3f}%)")
print(f"Multi-map:  {multi:,}  ({multi/total*100:.3f}%)")

# (Optional) If you need a bgzip file for tabix indexing:
#   !bgzip -c SCHEMA_variant_results_hg38.tsv.gz > SCHEMA_variant_results_hg38.tsv.bgz
# then:
#   !tabix -s 1 -b 2 -e 2 SCHEMA_variant_results_hg38.tsv.bgz


Lifting SCHEMA → hg38: 0it [00:00, ?it/s]/tmp/ipykernel_174014/1390401095.py:71: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  chunk.loc[needs_chr, "chr_hg19"] = "chr" + chunk.loc[needs_chr, "chr_hg19"].astype("string")
Lifting SCHEMA → hg38: 1it [00:09,  9.05s/it]/tmp/ipykernel_174014/1390401095.py:71: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  chunk.loc[needs_chr, "chr_hg19"] = "chr" + chunk.loc[needs_chr, "chr_hg19"].astype("string")
Lifting SCHEMA → hg38: 2it [00:17,  8.60s/it]/tmp/ipykernel_174014/1390401095.py:71: DeprecationWarnin

Lifting SCHEMA → hg38: 19it [02:43,  8.60s/it]/tmp/ipykernel_174014/1390401095.py:71: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  chunk.loc[needs_chr, "chr_hg19"] = "chr" + chunk.loc[needs_chr, "chr_hg19"].astype("string")
Lifting SCHEMA → hg38: 20it [02:52,  8.68s/it]/tmp/ipykernel_174014/1390401095.py:71: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  chunk.loc[needs_chr, "chr_hg19"] = "chr" + chunk.loc[needs_chr, "chr_hg19"].astype("string")
Lifting SCHEMA → hg38: 21it [03:00,  8.66s/it]/tmp/ipykernel_174014/1390401095.py:71: Deprecatio

Lifting SCHEMA → hg38: 38it [05:29,  8.67s/it]/tmp/ipykernel_174014/1390401095.py:71: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  chunk.loc[needs_chr, "chr_hg19"] = "chr" + chunk.loc[needs_chr, "chr_hg19"].astype("string")
Lifting SCHEMA → hg38: 39it [05:37,  8.61s/it]/tmp/ipykernel_174014/1390401095.py:71: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  chunk.loc[needs_chr, "chr_hg19"] = "chr" + chunk.loc[needs_chr, "chr_hg19"].astype("string")
Lifting SCHEMA → hg38: 40it [05:46,  8.57s/it]/tmp/ipykernel_174014/1390401095.py:71: Deprecatio

Lifting SCHEMA → hg38: 57it [08:11,  8.50s/it]/tmp/ipykernel_174014/1390401095.py:71: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  chunk.loc[needs_chr, "chr_hg19"] = "chr" + chunk.loc[needs_chr, "chr_hg19"].astype("string")
Lifting SCHEMA → hg38: 58it [08:19,  8.53s/it]/tmp/ipykernel_174014/1390401095.py:71: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  chunk.loc[needs_chr, "chr_hg19"] = "chr" + chunk.loc[needs_chr, "chr_hg19"].astype("string")
Lifting SCHEMA → hg38: 59it [08:28,  8.53s/it]/tmp/ipykernel_174014/1390401095.py:71: Deprecatio

Lifting SCHEMA → hg38: 76it [10:55,  8.63s/it]/tmp/ipykernel_174014/1390401095.py:71: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  chunk.loc[needs_chr, "chr_hg19"] = "chr" + chunk.loc[needs_chr, "chr_hg19"].astype("string")
Lifting SCHEMA → hg38: 77it [11:03,  8.62s/it]/tmp/ipykernel_174014/1390401095.py:71: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  chunk.loc[needs_chr, "chr_hg19"] = "chr" + chunk.loc[needs_chr, "chr_hg19"].astype("string")
Lifting SCHEMA → hg38: 78it [11:12,  8.64s/it]/tmp/ipykernel_174014/1390401095.py:71: Deprecatio

Lifting SCHEMA → hg38: 95it [13:37,  8.50s/it]/tmp/ipykernel_174014/1390401095.py:71: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  chunk.loc[needs_chr, "chr_hg19"] = "chr" + chunk.loc[needs_chr, "chr_hg19"].astype("string")
Lifting SCHEMA → hg38: 96it [13:46,  8.58s/it]/tmp/ipykernel_174014/1390401095.py:71: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  chunk.loc[needs_chr, "chr_hg19"] = "chr" + chunk.loc[needs_chr, "chr_hg19"].astype("string")
Lifting SCHEMA → hg38: 97it [13:55,  8.59s/it]/tmp/ipykernel_174014/1390401095.py:71: Deprecatio

Lifting SCHEMA → hg38: 113it [16:14,  8.67s/it]/tmp/ipykernel_174014/1390401095.py:71: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  chunk.loc[needs_chr, "chr_hg19"] = "chr" + chunk.loc[needs_chr, "chr_hg19"].astype("string")
Lifting SCHEMA → hg38: 114it [16:23,  8.67s/it]/tmp/ipykernel_174014/1390401095.py:71: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  chunk.loc[needs_chr, "chr_hg19"] = "chr" + chunk.loc[needs_chr, "chr_hg19"].astype("string")
Lifting SCHEMA → hg38: 115it [16:31,  8.62s/it]/tmp/ipykernel_174014/1390401095.py:71: Depreca

Done → /oak/stanford/groups/mrivas/projects/wgs-constraint-llm/data/SCHEMA_variant_results_hg38.tsv.gz
Total rows: 23,087,953
Mapped:     23,085,488  (99.989%)
Unmapped:   2,465  (0.011%)
Multi-map:  0  (0.000%)
